## Naive Data Parallelism

Welcome to having 0 knowledge about Parallel Processing in general, 0 knowledge about CUDA, and 0 knowledge about Pytorch with CUDA!

So uh... once we get over this, all the rest of hte parallelism will be very easy, we first start with a distributed setup.

In [ ]:
import os
import contextlib
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.autograd import Variable

In [ ]:
# torchrun sets these environment variables automatically
world_size = int(os.environ["WORLD_SIZE"])   # total GPUs (2 in this case)
global_rank = int(os.environ["RANK"])          # this process's ID (0,1)
local_rank = int(os.environ["LOCAL_RANK"])    # which GPU on this machine

# Connect all 2 processes so they can talk to each other (GPU-to-GPU)
dist.init_process_group(backend="nccl", init_method="env://")

# Tell this process which GPU to use (cuda:0, cuda:1, etc)
torch.cuda.set_device(local_rank)
device = torch.device("cuda", local_rank)

# For pure data parallelism, all GPUs work together
dp_size = world_size
dp_group = dist.new_group(ranks=list(range(dp_size)))  # creates a named group [0,1] for all_reduce

we can't run any of that code here... in jypter notebook, but the logic remains.

Now we can make our model, notice here we have sent the model in it's entirety to all devices.

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
).to(device)             # move to this GPU (cuda:0, cuda:1, etc)

Now, we can initalize the data parallel process.

The concept of data parallelism is simple:
- 4 GPUs, each has the same model
- Each sees different data
- Average the gradients at the end

But Pytorch is messy... so we have a bunch of these random names.

In [ ]:
# DataParallel wrapper - syncs gradients after backward
class DataParallelNaive(nn.Module):
    
    def __init__(self, module, dp_group, dp_world_size):
        super().__init__()
        self.module = module
        self.dp_group = dp_group
        self.dp_world_size = dp_world_size

        # For each trainable parameter, register a hook that runs when gradient is ready
        for p in module.parameters():
            if p.requires_grad:
                p.register_post_accumulate_grad_hook(self._hook)

Now, what the heck is a hook? A hook is just a function that runs automatically at a specific moment.

In this case, `register_post_accumulate_grad_hook` allows us to run our specific function after this parameter's gradient is accumalated.

But what do we want from this hook? The all reduce operation! We need the average gradient from all the GPUs!

In [ ]:
# Hook function - does all_reduce to average gradients across GPUs
def _hook(self, grad):
    dist.all_reduce(grad, op=dist.ReduceOp.SUM, group=self.dp_group)
    grad /= self.dp_world_size
    return grad

To put it all together.

In [ ]:
class DataParallelNaive(nn.Module):
    def __init__(self, module, dp_group, dp_world_size):
        super().__init__()
        self.module = module
        self.dp_group = dp_group
        self.dp_world_size = dp_world_size
        
        # Register hook on each parameter - runs when gradient is ready
        for p in module.parameters():
            if p.requires_grad:
                p.register_post_accumulate_grad_hook(self._hook)
    
    # Hook function - does all_reduce to average gradients across GPUs
    def _hook(self, grad):
        dist.all_reduce(grad, op=dist.ReduceOp.SUM, group=self.dp_group)
        grad /= self.dp_world_size
        return grad
    
    # Forward just passes through to the wrapped module
    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

In [ ]:
# Wrap the model with DataParallel
dp_model = DataParallelNaive(model, dp_group, dp_size)

Now, we need to prepare different data for the different GPUs! Assume we have 2 GPUs

1. Load a global batch - say 16 samples
2. Split it across GPUs - each GPU gets 8 samples
3. Each GPU processes its slice (through sampling!)

Global batch: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
                
- GPU 0 gets: [0, 1, 2, 3, 4, 5, 6, 7]
- GPU 1 gets: [8, 9, 10, 11, 12, 13, 14, 15]

That's not sampling, but just that we get half half in some way for 2 GPUs.

In [ ]:
# Full global batch (16 samples)
global_input = torch.randn(16, 8).to(device)
global_target = torch.randn(16, 4).to(device)

# Split both across GPUs
input_slice = global_input.chunk(dp_size, dim=0)[global_rank]   # 8 samples per GPU
target_slice = global_target.chunk(dp_size, dim=0)[global_rank] # 8 targets per GPU

We can now run a training step.

In [ ]:
optimizer = torch.optim.SGD(dp_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

output = dp_model(input_slice)
loss = loss_fn(output, target_slice)

loss.backward()  # gradients sync via hook
optimizer.step()
optimizer.zero_grad()

A full training loop therefore looks like something like this, the dataset is made as normal.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler


class MyDataset(Dataset):
    def __init__(self, size=160):
        self.data = torch.randn(size, 8)
        self.target = torch.randn(size, 4)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.target[idx]
    

mydataset = MyDataset()

But now... Pytorch automatically has a distributed sampler made for us! So we won't have to worry about splitting the different batches to different GPUs problem.

Uh... 'the respective GPU's rank' comes from the fact that, when different GPUs in the same distributed system runs the script, their `global rank` number will be different.

In [ ]:
# Key: dataset is SPLIT across GPUs
sampler = DistributedSampler(
    mydataset,
    num_replicas=dp_size,  # split across 2 GPUs
    rank=global_rank,       # each GPU gets different portion
    shuffle=True
)

# Each GPU automatically gets its slice through the sampler
dataloader = DataLoader(
    mydataset,
    batch_size=8,  # per-GPU batch size
    sampler=sampler,
    shuffle=False
)

In [ ]:
#full dataset
global_input = torch.randn(160, 8).to(device)
global_target = torch.randn(160, 4).to(device)

batch_size = 16  # global batch size
num_epochs = 4
optimizer = torch.optim.SGD(dp_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

In [ ]:
num_epochs = 4
batch_size = 16 # global batch size

dataloader = DataLoader(
    mydataset,
    batch_size=batch_size,  
    num_workers=0
)


for epoch in range(num_epochs):
    for batch in dataloader:
        global_input, global_target = batch
        
        # Move to GPUs and chunk like before
        global_input = global_input.to(device)
        global_target = global_target.to(device)
        
        input_slice = global_input.chunk(dp_size, dim=0)[global_rank]
        target_slice = global_target.chunk(dp_size, dim=0)[global_rank]
        
        # Forward → Loss → Backward → Optimizer
        output = dp_model(input_slice)
        loss = nn.MSELoss()(output, target_slice)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

So, that's technically the naive data parallism for you, this does however raise a question.

Say if our model has 1000 layers instead, the for every layer, should we do a  all_reduce? Not a great idea, we can instead gather these all_reduces into 'buckets' and do fewer number of all reudces on more layers.

As doing 1000 small all_reduces is much slower than doing 10 large ones! Due to the individual latency on each one.

## Bucket

A bucket is just one flat tensor that stores gradients for multiple parameters.

Without buckets:
- each parameter gradient is communicated separately

With a bucket:
- many parameter gradients are packed into one larger buffer
- then that one larger buffer is synchronized

Why this matters:
- fewer all_reduce calls
- bigger messages
- less communication overhead

That big flat tensor is the bucket.


In [ ]:
import torch

g1 = torch.tensor([1.0, 2.0])
g2 = torch.tensor([3.0, 4.0, 5.0])
g3 = torch.tensor([6.0])

bucket = torch.zeros(6)

bucket[0:2] = g1
bucket[2:5] = g2
bucket[5:6] = g3

print(bucket)

Later, each param can look back into its slice

In [ ]:
g1_view = bucket[0:2]
g2_view = bucket[2:5]
g3_view = bucket[5:6]

print(g2_view)

Once the idea of one bucket is clear, the bucket manager's job is just:

1. decide which parameters go into which bucket
2. assign each parameter a slice
3. create the flat bucket tensor
4. know when a full bucket is ready
5. launch one all_reduce for that bucket

We can turn this into a small class

In [2]:
import torch
import torch.distributed as dist

class Bucket:
    def __init__(self, params, grad_data, process_group):
        self.params = set(params)
        self.params_with_grad_ready = set()
        self.grad_data = grad_data
        self.process_group = process_group
        self.process_group_size = dist.get_world_size(group=process_group) if dist.is_initialized() else 1
        self.handle = None
        self.reset()

    def sync_gradient(self):
        self.grad_data /= self.process_group_size
        if dist.is_initialized() and self.process_group_size > 1:
            self.handle = dist.all_reduce(self.grad_data, group=self.process_group, async_op=True)

    def mark_param_as_ready(self, param):
        self.params_with_grad_ready.add(param)
        if len(self.params_with_grad_ready) == len(self.params):
            self.sync_gradient()

    def wait(self):
        if self.handle is not None:
            self.handle.wait()

    def reset(self):
        self.handle = None
        self.params_with_grad_ready.clear()
        self.grad_data.zero_()

In [ ]:
grad_data = torch.zeros(6)
bucket = Bucket(params=["w1", "w2", "w3"], grad_data=grad_data, process_group=None)

bucket.grad_data[0:2] = torch.tensor([1.0, 2.0])
bucket.grad_data[2:5] = torch.tensor([3.0, 4.0, 5.0])
bucket.grad_data[5:6] = torch.tensor([6.0])

bucket.mark_param_as_ready("w1")
bucket.mark_param_as_ready("w2")
bucket.mark_param_as_ready("w3")

print(bucket.grad_data)
print(bucket.params)
print(bucket.params_with_grad_ready == bucket.params)

By packing them into buckets and waiting for the bucket to fill, we can launch fewer, larger all_reduce operations.

## Bucket Manager

A BucketManager should take many parameter gradients and group them into several buckets.

Each bucket should have:
- a flat gradient buffer
- a list of params
- a mapping from param -> slice in the flat buffer

In practice, the sequential cap-based approach is what you'll usually see for gradient buckets, not a globally balanced greedy sort.


In [4]:
# Simulate bucket assignment logic using our Bucket class
bucket_cap = 100  # elements
params = [
    ("layer1.weight", 50),
    ("layer1.bias", 10),
    ("layer2.weight", 80),
    ("layer2.bias", 20),
    ("layer3.weight", 100),
    ("layer3.bias", 30),
]

print(f"Bucket capacity: {bucket_cap} elements\n")

buckets = []
current_bucket = []
current_bucket_size = 0

for name, numel in params:

    # If this param would overflow the current bucket, start a new one
    if current_bucket_size + numel > bucket_cap and current_bucket_size > 0:
        buckets.append(current_bucket)
        current_bucket = []
        current_bucket_size = 0

    current_bucket.append((name, numel))
    current_bucket_size += numel

    print(f"Added {name:20s} ({numel:3d} elements) to bucket {len(buckets)}")

# Don't forget the last bucket
if current_bucket:
    buckets.append(current_bucket)
print("\nFinal bucket assignment:")

for i, bucket in enumerate(buckets):
    total = sum(numel for _, numel in bucket)
    print(f"  Bucket {i}: {total:3d} elements - {[name for name, _ in bucket]}")

Bucket capacity: 100 elements

Added layer1.weight        ( 50 elements) to bucket 0
Added layer1.bias          ( 10 elements) to bucket 0
Added layer2.weight        ( 80 elements) to bucket 1
Added layer2.bias          ( 20 elements) to bucket 1
Added layer3.weight        (100 elements) to bucket 2
Added layer3.bias          ( 30 elements) to bucket 3

Final bucket assignment:
  Bucket 0:  60 elements - ['layer1.weight', 'layer1.bias']
  Bucket 1: 100 elements - ['layer2.weight', 'layer2.bias']
  Bucket 2: 100 elements - ['layer3.weight']
  Bucket 3:  30 elements - ['layer3.bias']


Now, this idea can again be turned into a class to manage buckets after buckets.

In [ ]:
class BucketManager:


    def __init__(self, params, process_group, bucket_size, grad_type=torch.float32):

        self.params = list(params)  # Convert generator to list
        self.device = self.params[0].device if self.params else torch.device('cpu')
        self.process_group = process_group
        self.bucket_size = bucket_size
        self.grad_type = grad_type
        
        # Core data structures
        self.params_to_bucket_location = {}  # param -> (start, end, bucket_idx)
        self.grad_data_list = []  # One flat tensor per bucket
        self.buckets = []
        
        self._initialize_buckets()
    
    def _initialize_buckets(self):
        """Divide parameters into buckets"""
        cur_bucket_size = 0
        cur_bucket_idx = 0
        
        # First pass: assign params to buckets
        for param in self.params:
            if not param.requires_grad:
                continue
            
            n = param.numel()
            
            # Start new bucket if empty
            if cur_bucket_size == 0:
                self.params_to_bucket_location[param] = (0, n, cur_bucket_idx)
                cur_bucket_size = n
            # Start new bucket if this param won't fit
            elif cur_bucket_size + n > self.bucket_size:
                cur_bucket_idx += 1
                self.params_to_bucket_location[param] = (0, n, cur_bucket_idx)
                cur_bucket_size = n
            # Add to current bucket
            else:
                self.params_to_bucket_location[param] = (
                    cur_bucket_size, cur_bucket_size + n, cur_bucket_idx
                )
                cur_bucket_size += n
        
        if not self.params_to_bucket_location:
            return
        
        # Second pass: compute bucket sizes and params per bucket
        num_buckets = max(idx for (_, _, idx) in self.params_to_bucket_location.values()) + 1
        bucket_sizes = [0] * num_buckets
        buckets_to_params = [[] for _ in range(num_buckets)]
        
        for param, (_, end, idx) in self.params_to_bucket_location.items():
            bucket_sizes[idx] = max(bucket_sizes[idx], end)
            buckets_to_params[idx].append(param)
        
        # Create flat grad tensors and Bucket objects
        for i in range(num_buckets):
            grad_data = torch.zeros(bucket_sizes[i], dtype=self.grad_type, device=self.device)
            self.grad_data_list.append(grad_data)
            self.buckets.append(Bucket(buckets_to_params[i], grad_data, self.process_group))
        
        # Create param.main_grad views (reverse order for compatibility)
        for param in self.params[::-1]:
            if not param.requires_grad:
                continue
            start, end, bucket_id = self.params_to_bucket_location[param]
            # main_grad is a view into the bucket tensor
            param.main_grad = self.grad_data_list[bucket_id][start:end].view(param.shape)
    
    def mark_param_as_ready(self, param):
        """Route mark_ready to the correct bucket"""
        bucket_idx = self.params_to_bucket_location[param][2]
        self.buckets[bucket_idx].mark_param_as_ready(param)
    
    def wait(self):
        """Wait for all buckets"""
        for bucket in self.buckets:
            bucket.wait()
    
    def reset(self):
        """Reset all buckets"""
        for bucket in self.buckets:
            bucket.reset()

## The Bucket Data Parallel

With the introduction of a bucket and bucket managers, we can move on from the naive data parallism, to dp with buckets.

In [ ]:
class DataParallelBucket(nn.Module):

    def __init__(self, module, process_group, bucket_size, grad_type=torch.float32):
        super().__init__()
        self.module = module
        self.process_group = process_group
        self.require_backward_grad_sync = True
        self.bucket_manager = BucketManager(
            module.parameters(),
            process_group=process_group,
            bucket_size=bucket_size,
            grad_type=grad_type,
        )
        self._post_backward_callback_set = False
        self._register_backward_hooks()

    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

    def _register_backward_hooks(self):
        self.grad_accs = []
        for param in self.module.parameters():
            if not param.requires_grad:
                continue
            param_tmp = param.expand_as(param)
            grad_acc_fn = param_tmp.grad_fn.next_functions[0][0]
            grad_acc_fn.register_hook(self._make_param_hook(param))
            self.grad_accs.append(grad_acc_fn)

    def _make_param_hook(self, param):
        def param_hook(*unused):
            if not param.requires_grad or param.grad is None:
                return

            param.main_grad.add_(param.grad.data)
            param.grad = None

            if not self.require_backward_grad_sync:
                return

            if not self._post_backward_callback_set:
                Variable._execution_engine.queue_callback(self._post_backward)
                self._post_backward_callback_set = True

            self.bucket_manager.mark_param_as_ready(param)

        return param_hook

    def _post_backward(self):
        self.bucket_manager.wait()
        self._post_backward_callback_set = False
        for param in self.module.parameters():
            if param.requires_grad:
                param.grad = param.main_grad.to(param.dtype)

    @contextlib.contextmanager
    def no_sync(self):
        self.require_backward_grad_sync = False
        yield
        self.require_backward_grad_sync = True

    def reset(self):
        self.bucket_manager.reset()

## Data Parallism On Mini GPT

Data parallelism wraps the whole MiniGPT model. Each rank keeps the full model, runs a different slice of the batch, and synchronizes gradients during backward.

In [ ]:
from mini_gpt import MiniTransformer, ModelConfig
import torch.nn.functional as F

In [ ]:
mini_gpt = MiniTransformer(ModelConfig()).to(device)
mini_gpt_dp = DataParallelBucket(
    mini_gpt,
    process_group=dp_group,
    bucket_size=100_000,
)

optimizer = torch.optim.AdamW(mini_gpt_dp.parameters(), lr=1e-3)

In [ ]:
batch_size = 8
seq_len = 16
vocab_size = mini_gpt.embedding.num_embeddings

input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
targets = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

logits = mini_gpt_dp(input_ids)
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

loss.backward()
optimizer.step()
optimizer.zero_grad()
mini_gpt_dp.reset()

print(f'MiniGPT bucketed DP step complete, loss={loss.item():.4f}')